In [1]:
# Cell 1: Clone repo + install deps
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
%cd cibuco-boriken
!pip install -q -r requirements.txt
print("Setup complete")

Cloning into 'cibuco-boriken'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 150 (delta 74), reused 106 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 263.72 KiB | 2.61 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/cibuco-boriken
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 157.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 159.6 MB/s e

In [2]:

!git pull origin main

# Verify
!python -c "
import subprocess
result = subprocess.run(['python', '-m', 'birdclef.train', '--help'],
capture_output=True, text=True)
print(result.stdout)



From https://github.com/drosadocastro-bit/cibuco-boriken
 * branch            main       -> FETCH_HEAD
Already up to date.
/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file
usage: train.py [-h]
                [--backbone {small,efficientnet_b0,efficientnet_b2,mobilenet_v2,convnext_tiny,perch}]
                [--epochs EPOCHS] [--batch-size BATCH_SIZE] [--lr LR]
                [--loss {bce,focal}] [--no-mixup] [--fast]
                [--max-samples MAX_SAMPLES] [--include-soundscapes]
                [--no-weighted-bce] [--smart-crop SMART_CROP]
                [--secondary-weight SECONDARY_WEIGHT]
                [--num-workers NUM_WORKERS] [--no-amp]
                [--precomputed PRECOMPUTED] [--patience PATIENCE]

BirdCLEF 2026 Training

options:
  -h, --help            show this help message and exit
  --backbone {small,efficientnet_b0,efficientnet_b2,mobilenet_v2,convnext_tiny,perch}
  --epochs EPOC

In [3]:
# Cell 2: Kaggle credentials (secure)
import os
import json
from google.colab import userdata

# Store token in Colab Secrets (never in code)
# Steps:
#   1. Click the 🔑 key icon in left sidebar
#   2. Add secret name: KAGGLE_USERNAME → drosadocastro-bit
#   3. Add secret name: KAGGLE_KEY → your_new_token
#   4. Enable notebook access for both

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_creds = {
    "username": userdata.get('kaggle_username'),
    "key": userdata.get('kaggle_key')
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle configured securely ✅")

# Download data
!pip install -q kaggle==1.6.17
!kaggle competitions download -c birdclef-2026
!mkdir -p data/birdclef-2026
!unzip -q birdclef-2026.zip -d data/birdclef-2026
print("Data ready ✅")


Kaggle configured securely ✅
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
100% 15.0G/15.0G [01:08<00:00, 245MB/s]
100% 15.0G/15.0G [01:08<00:00, 234MB/s]
Data ready ✅


In [5]:
# Cell 3: Mount Drive (save model after training)
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/cibuco_boriken/'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Drive mounted. Saving to: {SAVE_DIR}")

Mounted at /content/drive
Drive mounted. Saving to: /content/drive/MyDrive/cibuco_boriken/


In [6]:
import os

# Set BEFORE any birdclef imports
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

# Force reload entire birdclef module
import importlib
import birdclef
import birdclef.config as cfg

importlib.reload(cfg)

# Manually override if reload didn't catch it
from pathlib import Path
cfg.COMPETITION_DIR = Path(os.environ['BIRDCLEF_DATA_DIR'])
cfg.TRAIN_AUDIO_DIR = cfg.COMPETITION_DIR / "train_audio"
cfg.TEST_AUDIO_DIR  = cfg.COMPETITION_DIR / "test_soundscapes"
cfg.TRAIN_META_CSV  = cfg.COMPETITION_DIR / "train.csv"
cfg.TAXONOMY_CSV    = cfg.COMPETITION_DIR / "taxonomy.csv"
cfg.SAMPLE_SUBMISSION = cfg.COMPETITION_DIR / "sample_submission.csv"

# Verify
print(f"Data dir: {cfg.COMPETITION_DIR}")
print(f"Train CSV exists: {cfg.TRAIN_META_CSV.exists()}")
print(f"Train audio exists: {cfg.TRAIN_AUDIO_DIR.exists()}")
print(f"Soundscapes exists: {cfg.COMPETITION_DIR / 'train_soundscapes_labels.csv'}")

Data dir: /content/cibuco-boriken/data/birdclef-2026
Train CSV exists: True
Train audio exists: True
Soundscapes exists: /content/cibuco-boriken/data/birdclef-2026/train_soundscapes_labels.csv


In [7]:
import os
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

# Verify env var is set
print(os.environ.get('BIRDCLEF_DATA_DIR'))

# Check file directly without config
from pathlib import Path
p = Path('/content/cibuco-boriken/data/birdclef-2026/train.csv')
print(f"File exists directly: {p.exists()}")

/content/cibuco-boriken/data/birdclef-2026
File exists directly: True


In [8]:
!ls /content/cibuco-boriken/data/birdclef-2026/*.csv

/content/cibuco-boriken/data/birdclef-2026/sample_submission.csv
/content/cibuco-boriken/data/birdclef-2026/taxonomy.csv
/content/cibuco-boriken/data/birdclef-2026/train.csv
/content/cibuco-boriken/data/birdclef-2026/train_soundscapes_labels.csv


In [9]:
!python -m birdclef.smart_crop --train-csv /content/cibuco-boriken/data/birdclef-2026/train.csv
!python -m birdclef.precompute --manifest birdclef/output/smart_crop_manifest.csv
!python -m birdclef.train --backbone convnext_tiny --precomputed /content/cibuco-boriken/birdclef/output/precomputed --epochs 30 --batch-size 64 --patience 7

birdclef.smart_crop | INFO | Processed 500 / 35549 files  (smart: 1288 / naive: 2576 windows)
birdclef.smart_crop | INFO | Processed 1000 / 35549 files  (smart: 2699 / naive: 5399 windows)
birdclef.smart_crop | INFO | Processed 1500 / 35549 files  (smart: 4812 / naive: 9717 windows)
birdclef.smart_crop | INFO | Processed 2000 / 35549 files  (smart: 6353 / naive: 12797 windows)
birdclef.smart_crop | INFO | Processed 2500 / 35549 files  (smart: 8478 / naive: 16973 windows)
birdclef.smart_crop | INFO | Processed 3000 / 35549 files  (smart: 10012 / naive: 19982 windows)
birdclef.smart_crop | INFO | Processed 3500 / 35549 files  (smart: 11442 / naive: 22768 windows)
birdclef.smart_crop | INFO | Processed 4000 / 35549 files  (smart: 13107 / naive: 25993 windows)
birdclef.smart_crop | INFO | Processed 4500 / 35549 files  (smart: 15146 / naive: 30149 windows)
birdclef.smart_crop | INFO | Processed 5000 / 35549 files  (smart: 16593 / naive: 33063 windows)
birdclef.smart_crop | INFO | Processed 

In [10]:
# SAVE MODEL IMMEDIATELY AFTER TRAINING
import shutil
shutil.copy(
  '/content/cibuco-boriken/birdclef/models/birdclef_model.pt',
  SAVE_DIR + 'convnext_tiny_SMARTCROP_BCE_20260316_v1.pt'
)
print(f"Model saved to Drive ✅")
print(f"File size: {os.path.getsize(SAVE_DIR + 'convnext_tiny_SMARTCROP_BCE_20260316_v1.pt') / 1e6:.1f} MB")

Model saved to Drive ✅
File size: 112.0 MB


In [ ]:
# Cell 5: CFAR Phase 2 k-sweep (3 conditions)
import os
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

!BIRDCLEF_DATA_DIR=/content/cibuco-boriken/data/birdclef-2026 \
  python -m birdclef.evaluate_thresholds \
  --backbone efficientnet_b2 \
  --include-soundscapes \
  --k-sweep 1.0 2.0 3.0 \
  --temperature 0.3

print("k-sweep complete ✅")

In [ ]:
# Cell 6: Display and save figure/model
from IPython.display import Image, display
import shutil

display(Image('k_sweep_figure.png'))

shutil.copy('k_sweep_figure.png', SAVE_DIR + 'k_sweep_500samples.png')
shutil.copy('birdclef/models/birdclef_model.pt', SAVE_DIR + 'birdclef_model_500samples.pt')
print("Saved to Drive")

In [ ]:
# Cell 7: Print paper-ready results table
import json
from pathlib import Path

results_path = Path('k_sweep_results.json')
if not results_path.exists():
    print('k_sweep_results.json not found. Run Cell 5 first.')
else:
    rows = json.loads(results_path.read_text(encoding='utf-8'))
    print('## Table 1: CFAR k-Sensitivity Results (500 samples)')
    print('| k | F1 Fixed | F1 CFAR | FPR Fixed | FPR CFAR | T_mean |')
    print('|---|----------|---------|-----------|----------|--------|')
    for r in rows:
        print(f"| {r['k']:.1f} | {r['f1_fixed']:.4f} | {r['f1_cfar']:.4f} | {r['fpr_fixed']:.4f} | {r['fpr_cfar']:.4f} | {r['threshold_mean']:.4f} |")

In [ ]:
# Temperature Goldilocks curve
import matplotlib.pyplot as plt
import numpy as np

temps  = [0.05,   0.10,   0.20,   0.30,   0.50]
deltas = [0.0001, 0.0003, 0.0012, 0.0019, -0.0394]
aucs   = [0.5934, 0.7525, 0.7582, 0.7582, 0.7582]
fprs   = [0.0003, 0.0003, 0.0004, 0.0006, 0.0011]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 2: Temperature Scaling — Goldilocks Zone for CFAR')

# Left: F1 delta vs temperature
ax1.plot(temps, deltas, 'b-o', linewidth=2, markersize=8)
ax1.axhline(y=0, color='gray', linestyle='--', label='Fixed baseline')
ax1.axvline(x=0.3, color='green', linestyle='--', alpha=0.7, label='Optimal T=0.3')
ax1.fill_between(temps, deltas, 0,
                  where=[d>0 for d in deltas],
                  alpha=0.2, color='green', label='CFAR advantage')
ax1.fill_between(temps, deltas, 0,
                  where=[d<0 for d in deltas],
                  alpha=0.2, color='red', label='CFAR disadvantage')
ax1.set_xlabel('Temperature T')
ax1.set_ylabel('F1 Delta (CFAR - Fixed)')
ax1.set_title('F1 Improvement vs Temperature')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: AUC vs temperature
ax2.plot(temps, aucs, 'r-s', linewidth=2, markersize=8)
ax2.axvline(x=0.3, color='green', linestyle='--', alpha=0.7, label='Optimal T=0.3')
ax2.set_xlabel('Temperature T')
ax2.set_ylabel('ROC-AUC')
ax2.set_title('AUC vs Temperature')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('temperature_goldilocks.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved ✅")